# SHIPP Stage 8 — Gold candidate matches (scoring v1)

`silver_candidate_pairs` + `silver_routes` + `silver_listings` + `silver_requests` → `gold_candidate_matches`
**Grain:** one row = one ranked candidate listing for one open request (top N per request).
Read directly by the Agent's `get_candidate_matches(request_id)` and by the App.

**Scoring contract** — `data_pipeline/gold/scoring.py`, weights in `config/settings.py → SCORING`:

```text
match_score = 0.50 × distance_score + 0.30 × timing_score + 0.20 × condition_score
distance_score  = max(0, 1 − route_km / 50)   (0 if no OK route — never invented)
timing_score    = min(1, slack_days / 7)
condition_score = NEW 1.0 · LIKE_NEW 0.9 · GOOD 0.75 · FAIR 0.5 · other 0.3
match_pct       = round(100 × match_score)
```
v2 adds `semantic_score` from AI Search (weights noted in settings). **Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of this table.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
from data_pipeline.gold import scoring

scoring.assert_weights_valid()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

EXPECTED_COLUMNS = [
    "request_id", "listing_id", "match_rank", "match_score", "match_pct",
    "distance_score", "timing_score", "condition_score",
    "route_status", "route_distance_km", "route_duration_min", "straight_line_km",
    "category", "listing_title", "listing_description", "listing_condition", "listing_status",
    "available_from", "available_until", "need_by_date", "request_text",
    "scoring_version", "gold_processed_at",
]

pairs = spark.table(SILVER_CANDIDATE_PAIRS).alias("p")
routes = (spark.table(SILVER_ROUTES) if table_exists(SILVER_ROUTES) else spark.createDataFrame(
    [], "origin_key string, dest_key string, profile string, distance_km double, duration_min double, route_status string")).alias("rt")
listings = spark.table(SILVER_LISTINGS).filter(F.col("status").isin(ELIGIBLE_LISTING_STATUSES)).alias("l")
requests_df = spark.table(SILVER_REQUESTS).filter(F.col("status").isin(ELIGIBLE_REQUEST_STATUSES)).alias("r")
now = now_ntz()

joined = (
    pairs
    .join(routes, ["origin_key", "dest_key", "profile"], "left")
    .join(listings, F.col("p.listing_id") == F.col("l.listing_id"), "inner")
    .join(requests_df, F.col("p.request_id") == F.col("r.request_id"), "inner")
    .select(
        F.col("p.request_id").alias("request_id"), F.col("p.listing_id").alias("listing_id"),
        F.coalesce(F.col("rt.route_status"), F.lit("PENDING")).alias("route_status"),
        F.when(F.col("rt.route_status") == "OK", F.col("rt.distance_km")).alias("route_distance_km"),
        F.when(F.col("rt.route_status") == "OK", F.col("rt.duration_min")).alias("route_duration_min"),
        F.col("p.straight_line_km"), F.col("p.category"),
        F.col("l.title").alias("listing_title"), F.col("l.description").alias("listing_description"),
        F.col("l.condition").alias("listing_condition"), F.col("l.status").alias("listing_status"),
        F.col("l.available_from"), F.col("l.available_until"),
        F.col("r.need_by_date"), F.col("r.request_text"),
    )
)

scored = (
    joined
    .withColumn("distance_score", F.round(scoring.distance_score(F.col("route_status"), F.col("route_distance_km")), 4))
    .withColumn("timing_score", F.round(scoring.timing_score(F.col("available_from"), F.col("need_by_date"), now), 4))
    .withColumn("condition_score", scoring.condition_score(F.col("listing_condition")))
    .withColumn("match_score", scoring.match_score(F.col("distance_score"), F.col("timing_score"), F.col("condition_score")))
)
rank = Window.partitionBy("request_id").orderBy(
    F.col("match_score").desc(), F.col("route_distance_km").asc_nulls_last(), F.col("listing_id"))
gold_df = (
    scored
    .withColumn("match_rank", F.row_number().over(rank))
    .filter(F.col("match_rank") <= SCORING["top_n_per_request"])
    .withColumn("match_pct", F.round(F.col("match_score") * 100).cast("int"))
    .withColumn("scoring_version", F.lit(SCORING["version"]))
    .withColumn("gold_processed_at", F.current_timestamp())
    .select(*EXPECTED_COLUMNS)
)
row_count = gold_df.count()
display(gold_df.select("request_id", "match_rank", "listing_id", "listing_title", "match_pct",
                       "route_status", "route_distance_km", "route_duration_min").orderBy("request_id", "match_rank"))

## Quality gate — the Agent is blocked until this passes

In [ ]:
run_quality_gate({
    "duplicate (request, listing)": duplicate_key_count(gold_df, "request_id", "listing_id"),
    "duplicate (request, rank)": duplicate_key_count(gold_df, "request_id", "match_rank"),
    "score outside [0,1]": gold_df.filter(~F.col("match_score").between(0, 1)).count(),
    "listing not eligible": gold_df.filter(~F.col("listing_status").isin(ELIGIBLE_LISTING_STATUSES)).count(),
    "distance without OK route": gold_df.filter((F.col("route_status") != "OK") & F.col("route_distance_km").isNotNull()).count(),
    "OK route without distance": gold_df.filter((F.col("route_status") == "OK") & F.col("route_distance_km").isNull()).count(),
}, row_count=row_count, allow_empty=True)

pending = gold_df.filter(F.col("route_status") == "PENDING").count()
if pending:
    print(f"WARNING — {pending} match(es) without a route yet; rerun 31 + 32.")

written = write_overwrite(gold_df, GOLD_CANDIDATE_MATCHES)
assert written == row_count, f"WRITE VALIDATION FAILED: wrote {written}, expected {row_count}"
assert_exact_schema(GOLD_CANDIDATE_MATCHES, EXPECTED_COLUMNS)

summary = {"table": GOLD_CANDIDATE_MATCHES, "rows": written, "pending_routes": pending,
           "requests_with_matches": spark.table(GOLD_CANDIDATE_MATCHES).select("request_id").distinct().count(),
           "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))